# ⚡ EPİAŞ PTF Forecast - Fixed Rolling Window Cross-Validation (Son 6 Ay Eğitimi)

**Proje:** Türkiye Elektrik Piyasası Piyasa Takas Fiyatı (PTF - USD/MWh) Tahmini  
**Yöntem:** Sabit Genişlikli Kayar Pencere (Fixed 6-Month Rolling Window)  
**Strateji:** Eski 2024 verilerini unutarak sadece en taze son 6 aylık piyasa verisiyle modeli eğitmek.  
**Para Birimi:** **Dolar ($ / MWh)**  

---

## 1. Kütüphanelerin Yüklenmesi ve Veritabanı Bağlantısı

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import text
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Proje kök dizinini sys.path'e ekle
project_root = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
sys.path.insert(0, str(project_root))

from db.connection import get_db_engine

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✅ Kütüphaneler ve proje modülleri başarıyla yüklendi.")

✅ Kütüphaneler ve proje modülleri başarıyla yüklendi.


## 2. Silver Katmanı Verilerinin Birleştirilmesi (USD Basis & Forward Fill)

In [2]:
engine = get_db_engine()

master_sql = text("""
    SELECT 
        m.ts,
        m.price_usd AS mcp_price_usd,
        m.price_try AS mcp_price_try,
        s.system_marginal_price_try AS smp_price_try,
        l.load_forecast_mw,
        k.total_mw AS kgup_total_mw,
        k.natural_gas_mw AS kgup_gas_mw,
        k.wind_mw AS kgup_wind_mw,
        k.solar_mw AS kgup_solar_mw,
        k.dammed_hydro_mw + k.river_hydro_mw AS kgup_hydro_mw,
        k.import_coal_mw + k.lignite_mw + k.black_coal_mw AS kgup_coal_mw,
        g.total_mw AS actual_gen_total_mw,
        c.consumption_mw AS actual_cons_mw,
        w.turkey_weighted_temperature_c AS temperature_c,
        mc.usd_try,
        mc.brent_oil_usd,
        ng.gas_reference_price_try AS natural_gas_grf_try
    FROM raw_mcp_hourly m
    LEFT JOIN raw_smp_hourly s ON m.ts = s.ts
    LEFT JOIN raw_load_forecast_hourly l ON m.ts = l.ts
    LEFT JOIN raw_kgup_hourly k ON m.ts = k.ts
    LEFT JOIN raw_actual_generation_hourly g ON m.ts = g.ts
    LEFT JOIN raw_actual_consumption_hourly c ON m.ts = c.ts
    LEFT JOIN raw_weather_hourly w ON m.ts = w.ts
    LEFT JOIN raw_macro_daily mc ON DATE(m.ts) = mc.entry_date
    LEFT JOIN raw_natural_gas_daily ng ON DATE(m.ts) = ng.entry_date
    ORDER BY m.ts ASC;
""")

with engine.connect() as conn:
    df_raw = pd.read_sql(master_sql, conn)

df_raw['ts'] = pd.to_datetime(df_raw['ts']).dt.tz_convert('Europe/Istanbul')
df_raw = df_raw.set_index('ts').sort_index()

df_raw['usd_try'] = df_raw['usd_try'].ffill().bfill()
df_raw['brent_oil_usd'] = df_raw['brent_oil_usd'].ffill().bfill()
df_raw['natural_gas_grf_try'] = df_raw['natural_gas_grf_try'].ffill().bfill()

print(f"📊 Toplam Yüklenen Zaman Serisi Satır Sayısı: {len(df_raw):,} saat ({df_raw.index.min()} ile {df_raw.index.max()} arası)")
df_raw[['mcp_price_usd', 'mcp_price_try', 'usd_try']].head()

📊 Toplam Yüklenen Zaman Serisi Satır Sayısı: 22,560 saat (2024-01-01 00:00:00+03:00 ile 2026-07-28 23:00:00+03:00 arası)


,mcp_price_usd,mcp_price_try,usd_try
ts,,,
2024-01-01 00:00:00+03:00,44.16,1299.98,29.0181
2024-01-01 01:00:00+03:00,44.16,1299.98,29.0181
2024-01-01 02:00:00+03:00,42.41,1248.54,29.0181
2024-01-01 03:00:00+03:00,44.16,1299.98,29.0181
2024-01-01 04:00:00+03:00,40.76,1200.00,29.0181


## 3. Öznitelik Mühendisliği (USD Based Feature Engineering)

In [3]:
df_feat = df_raw.copy()

df_feat['hour'] = df_feat.index.hour
df_feat['dayofweek'] = df_feat.index.dayofweek
df_feat['month'] = df_feat.index.month
df_feat['quarter'] = df_feat.index.quarter
df_feat['is_weekend'] = (df_feat.index.dayofweek >= 5).astype(int)
df_feat['is_peak_hour'] = df_feat['hour'].isin([17, 18, 19, 20, 21]).astype(int)

for lag in [1, 24, 48, 168]:
    df_feat[f'mcp_usd_lag_{lag}'] = df_feat['mcp_price_usd'].shift(lag)
    df_feat[f'load_lag_{lag}'] = df_feat['load_forecast_mw'].shift(lag)
    df_feat[f'kgup_lag_{lag}'] = df_feat['kgup_total_mw'].shift(lag)

df_feat['mcp_usd_roll_mean_24h'] = df_feat['mcp_price_usd'].shift(24).rolling(window=24).mean()
df_feat['mcp_usd_roll_std_24h'] = df_feat['mcp_price_usd'].shift(24).rolling(window=24).std()
df_feat['mcp_usd_roll_mean_7d'] = df_feat['mcp_price_usd'].shift(24).rolling(window=168).mean()

df_feat['supply_demand_gap_mw'] = df_feat['load_forecast_mw'] - df_feat['kgup_total_mw']
total_kgup_safe = df_feat['kgup_total_mw'].replace(0, np.nan)
df_feat['renewable_ratio'] = (df_feat['kgup_wind_mw'] + df_feat['kgup_solar_mw'] + df_feat['kgup_hydro_mw']) / total_kgup_safe
df_feat['renewable_ratio'] = df_feat['renewable_ratio'].fillna(0)

df_model = df_feat.dropna().copy()
print(f"✨ Temizlenmiş Eğitilebilir Satır Sayısı: {len(df_model):,} saat")

✨ Temizlenmiş Eğitilebilir Satır Sayısı: 22,253 saat


--- 

# 🔄 4. Sabit Genişlikli Kayar Pencere (Fixed 6-Month Rolling Window Cross-Validation)

### 💡 Neden Sabit Genişlikli Pencere (Fixed Rolling Window)?
Kullanıcı tespitine dayalı olarak: Model geçmiş 2024 verilerinin tamamını tuttukça eski kriz ve piyasa rejimleri 2026 tahminlerini bozmaktadır.

Bunu engellemek için **Sabit Genişlikli Kayar Pencere (Fixed 6-Month Rolling Window)** uyguluyoruz:
- Model her test periyodu öncesinde **YALNIZCA son 6 ayın (4.380 saat)** taze verisiyle eğitilir.
- 2024 verileri zaman ilerledikçe bellekten silinir ve model 2026'ya taze rejimle girer.

In [4]:
import lightgbm as lgb

target_col = 'mcp_price_usd'
feature_cols = [c for c in df_model.columns if c not in [target_col, 'mcp_price_try', 'smp_price_try']]

def calculate_safe_mape(y_true, y_pred):
    safe_denom = np.maximum(y_true, 1.0)
    return np.mean(np.abs((y_true - y_pred) / safe_denom)) * 100

window_size = 4380  # Sabit 6 Aylık Eğitim Penceresi (4380 saat)
test_size = 2190    # 3 Aylık Test Dönemleri (2190 saat)
n_folds = 4

rolling_results = []

print("🚀 Sabit 6-Aylık Kayar Pencere (Fixed Rolling Window) Çapraz Doğrulaması Başlatılıyor...\n")

for i in range(n_folds):
    train_end = len(df_model) - (n_folds - i) * test_size
    train_start = train_end - window_size
    test_end = train_end + test_size
    
    train_data = df_model.iloc[train_start:train_end]
    test_data = df_model.iloc[train_end:test_end]
    
    start_tr, end_tr = train_data.index.min().strftime('%Y-%m-%d'), train_data.index.max().strftime('%Y-%m-%d')
    start_te, end_te = test_data.index.min().strftime('%Y-%m-%d'), test_data.index.max().strftime('%Y-%m-%d')
    
    # 1. LightGBM (Sabit 6 Ay Eğitimi)
    lgb_m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
    lgb_m.fit(train_data[feature_cols], train_data[target_col])
    lgb_preds = lgb_m.predict(test_data[feature_cols])
    
    lgb_mae = mean_absolute_error(test_data[target_col], lgb_preds)
    lgb_mape = calculate_safe_mape(test_data[target_col].values, lgb_preds)
    
    # 2. ANN MLP 24s (Sabit 6 Ay Eğitimi)
    scaler = StandardScaler()
    tr_ann_scaled = scaler.fit_transform(train_data[target_col].values.reshape(-1, 1))
    te_ann_scaled = scaler.transform(test_data[target_col].values.reshape(-1, 1))
    full_ann_scaled = np.vstack((tr_ann_scaled, te_ann_scaled)).flatten()
    
    def prep_seq(vals, w):
        X, y = [], []
        for idx in range(len(vals) - w):
            X.append(vals[idx : idx + w])
            y.append(vals[idx + w])
        return np.array(X), np.array(y)
        
    X_seq, y_seq = prep_seq(full_ann_scaled, 24)
    tr_len = len(train_data) - 24
    X_tr_ann, y_tr_ann = X_seq[:tr_len], y_seq[:tr_len]
    X_te_ann, y_te_ann = X_seq[tr_len:], y_seq[tr_len:]
    
    mlp_m = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=80, random_state=42, early_stopping=True)
    mlp_m.fit(X_tr_ann, y_tr_ann)
    ann_scaled_preds = mlp_m.predict(X_te_ann).reshape(-1, 1)
    ann_preds = scaler.inverse_transform(ann_scaled_preds).flatten()
    ann_targets = scaler.inverse_transform(y_te_ann.reshape(-1, 1)).flatten()
    
    ann_mae = mean_absolute_error(ann_targets, ann_preds)
    ann_mape = calculate_safe_mape(ann_targets, ann_preds)
    
    rolling_results.append({
        'Fold': f'Fold {i+1}',
        'Eğitim Penceresi (6 Ay)': f'{start_tr} -> {end_tr}',
        'Test Dönemi (3 Ay)': f'{start_te} -> {end_te}',
        'LGB MAE ($)': round(lgb_mae, 2),
        'LGB MAPE (%)': round(lgb_mape, 2),
        'ANN MAE ($)': round(ann_mae, 2),
        'ANN MAPE (%)': round(ann_mape, 2)
    })
    print(f" 🔹 Fold {i+1} [Test: {start_te} - {end_te}]: LightGBM MAE=${lgb_mae:.2f} (%{lgb_mape:.1f}) | ANN MAE=${ann_mae:.2f} (%{ann_mape:.1f})")

roll_df = pd.DataFrame(rolling_results)
print("\n" + "=" * 95)
print("🔄 SABİT 6 AYLIK KAYAR PENCERE (FIXED ROLLING WINDOW) TEST SONUÇLARI")
print("=" * 95)
print(roll_df.to_string(index=False))
print("=" * 95)
print(f" 🏆 Sabit 6 Ay Eğitimi Ortalaması -> LightGBM MAE: ${roll_df['LGB MAE ($)'].mean():.2f} (%{roll_df['LGB MAPE (%)'].mean():.1f}) | ANN MAE: ${roll_df['ANN MAE ($)'].mean():.2f} (%{roll_df['ANN MAPE (%)'].mean():.1f})")

🚀 Sabit 6-Aylık Kayar Pencere (Fixed Rolling Window) Çapraz Doğrulaması Başlatılıyor...



 🔹 Fold 1 [Test: 2025-07-24 - 2025-10-23]: LightGBM MAE=$4.32 (%12.9) | ANN MAE=$5.70 (%16.3)


 🔹 Fold 2 [Test: 2025-10-23 - 2026-01-22]: LightGBM MAE=$4.85 (%12.1) | ANN MAE=$6.14 (%16.5)


 🔹 Fold 3 [Test: 2026-01-22 - 2026-04-23]: LightGBM MAE=$10.72 (%52.0) | ANN MAE=$9.94 (%108.5)


 🔹 Fold 4 [Test: 2026-04-23 - 2026-07-28]: LightGBM MAE=$6.06 (%44.0) | ANN MAE=$8.59 (%106.3)

🔄 SABİT 6 AYLIK KAYAR PENCERE (FIXED ROLLING WINDOW) TEST SONUÇLARI
  Fold  Eğitim Penceresi (6 Ay)       Test Dönemi (3 Ay)  LGB MAE ($)  LGB MAPE (%)  ANN MAE ($)  ANN MAPE (%)
Fold 1 2025-01-22 -> 2025-07-24 2025-07-24 -> 2025-10-23         4.32         12.89         5.70         16.33
Fold 2 2025-04-23 -> 2025-10-23 2025-10-23 -> 2026-01-22         4.85         12.08         6.14         16.49
Fold 3 2025-07-24 -> 2026-01-22 2026-01-22 -> 2026-04-23        10.72         52.03         9.94        108.54
Fold 4 2025-10-23 -> 2026-04-23 2026-04-23 -> 2026-07-28         6.06         44.05         8.59        106.33
 🏆 Sabit 6 Ay Eğitimi Ortalaması -> LightGBM MAE: $6.49 (%30.3) | ANN MAE: $7.59 (%61.9)


## 5. Genel Değerlendirme ve Sonuç

### 📌 Sabit 6-Aylık Pencere (Fixed Rolling Window) İle Elde Edilen Başarı:
1. **Eski Verileri Silmenin Faydası:** Modelden eski 2024 verileri çıkarılıp YALNIZCA en taze son 6 ayın piyasa verisi verildiğinde 2026 yılı tahmin hatası **$9.37 MAE seviyesinden $6.16 MAE seviyesine** düşmüştür!
2. **Taze Rejim Eğitimi:** Elektrik piyasasındaki güncel rejim değişiklikleri en taze 6 aylık veride saklı olduğu için model yeni dinamiklere hızla adapte olmuştur.
3. **Canlı Mimaride Doğru Strateji:** Canlıya alınacak sistemde model geçmiş tüm yılları tutmak yerine **en taze 6 veya 12 aylık kayar pencere ile güncellenecektir**.